In [ ]:
import torch
from torch import nn

In [ ]:
class Classifier(nn.Module):
    """분류 모델들이 공통으로 사용할 기반 클래스."""

    def __init__(self, learning_rate=0.1):
        super().__init__()

        # learning rate(step size): 학습률
        self.learning_rate = learning_rate

    def forward(self, *inputs):
        # 실제 신경망 구조는 자식 클래스에서 구현한다.
        raise NotImplementedError

    def loss(self, logits, labels):
        # 실제 손실 계산은 자식 클래스에서 구현한다.
        raise NotImplementedError

    def accuracy(self, logits, labels):
        # 정확도 계산은 4.3.2에서 구현한다.
        raise NotImplementedError

    @torch.no_grad()
    def validation_step(self, batch):
        # 마지막 원소는 label이고,
        # 그 앞의 원소들은 모델 입력이다.
        *inputs, labels = batch

        # self(*inputs)는 내부적으로 forward를 호출한다.
        logits = self(*inputs)

        validation_loss = self.loss(
            logits,
            labels,
        )

        validation_accuracy = self.accuracy(
            logits,
            labels,
        )

        # 이후 전체 검증 결과를 집계할 수 있도록
        # 배치의 metric과 데이터 개수를 반환한다.
        return {
            "loss": validation_loss.item(),
            "accuracy": validation_accuracy.item(),
            "num_examples": labels.shape[0],
        }

    def configure_optimizer(self):
        # self.parameters()는 자식 모델이 가진
        # 학습 가능한 모든 parameter를 반환한다.
        return torch.optim.SGD(
            self.parameters(),
            lr=self.learning_rate,
        )

In [ ]:
# DataLoader가 반환하는 batch 구조를 작은 예제로 확인한다.
demo_features = torch.randn(4, 1, 32, 32)
demo_labels = torch.tensor([0, 3, 5, 9])

demo_batch = (
    demo_features,
    demo_labels,
)

*inputs, labels = demo_batch

print("Number of model inputs:", len(inputs))
print("Input shape:", inputs[0].shape)
print("Label shape:", labels.shape)

Number of model inputs: 1
Input shape: torch.Size([4, 1, 32, 32])
Label shape: torch.Size([4])


In [ ]:
# Cross-Entropy
# → 확률의 미세한 변화도 측정 가능
# → 미분 가능
# → 모델 학습에 사용

# Accuracy
# → 최종 클래스를 맞혔는지만 측정
# → 미분 불가능
# → 모델 성능 보고에 사용
def classifier_accuracy(
    self,
    logits,
    labels,
    averaged=True,
):
    """분류 모델의 정확도를 계산한다."""

    # 마지막 차원은 클래스 차원이다.
    # 그 앞의 모든 차원은 하나로 펼친다.
    #
    # 일반적인 경우:
    # (batch_size, num_classes)
    # -> (batch_size, num_classes)
    logits = logits.reshape(
        -1,
        logits.shape[-1],
    )

    # 각 데이터에서 가장 큰 logit의 위치를 선택한다.
    predictions = logits.argmax(dim=-1)

    # 예측 클래스와 실제 label의 dtype을 맞춘다.
    predictions = predictions.to(labels.dtype)

    # label도 1차원으로 펼친다.
    labels = labels.reshape(-1)

    # 올바른 예측은 True, 틀린 예측은 False다.
    correct = predictions == labels

    # 평균을 계산할 수 있도록 bool을 float로 변환한다.
    correct = correct.to(torch.float32)

    if averaged:
        return correct.mean()

    return correct


# D2L의 @add_to_class와 같은 역할이다.
# 이미 정의된 Classifier 클래스에 accuracy 메서드를 추가한다.
Classifier.accuracy = classifier_accuracy

In [ ]:
demo_logits = torch.tensor([
    [2.0, 1.0, 0.0],
    [0.0, 2.0, 1.0],
    [2.0, 0.0, 1.0],
])

demo_labels = torch.tensor([
    0,
    1,
    2,
])

classifier = Classifier()

predictions = demo_logits.argmax(dim=-1)

correct = classifier.accuracy(
    demo_logits,
    demo_labels,
    averaged=False,
)

accuracy = classifier.accuracy(
    demo_logits,
    demo_labels,
    averaged=True,
)

print("Predictions:", predictions)
print("Labels:", demo_labels)
print("Correct:", correct)
print("Accuracy:", accuracy)

Predictions: tensor([0, 1, 0])
Labels: tensor([0, 1, 2])
Correct: tensor([1., 1., 0.])
Accuracy: tensor(0.6667)
